---
# <div style="text-align: center"> Introduction </div>
---

Along these tutorials, we will see how <span style="color:blue">**SCOPE**</span> interacts with the different parts of the code to handle the execution of computational workflows. 

These are the topics covered in each tutorial:
1) The **System** class and its sources: the **Specie**, **Cell** and **Atom** classes  
2) The Computational workflow: **Branch**, **Workflow**, **Job**, and **Computation** classes  
3) The **State** class  
4) The **Data**, **Collection** and **VNM** classes
5) The **Input_data** class, and **scope input files**
6) Running <span style="color:blue">**SCOPE**</span> - Part 1: File Structure
7) Running <span style="color:blue">**SCOPE**</span> - Part 2: Execution 
8) Running <span style="color:blue">**SCOPE**</span> - Part 3: Detailed Actions
9) The Azo add-on: Subclasses, Analysis and Creation of Azo Systems

---
# <div style="text-align: center"> Tutorial 9: The Azo Add-on </div>
---

In this tutorial, we discuss the specific Classes introduced in the **Azo** add-on of **SCOPE**. It is highly recommended to follow the previous tutorials first.  
This tutorial is structured in 4 parts:

1) System and State Subclasses: **System_azo**, **State_azo**
2) Analysis of Thermal Properties.
3) Analysis of Optical Properties: **PSS** and **Lamp** classes
4) Creation of new Azo systems

# Introduction to Azo-based photoswitches

Azo-based photoswitches can reversibly interconvert between **trans (E)** and **cis (Z)** isomers.

- **Trans (E)** is usually the thermodynamically more stable form. 
- **Cis (Z)** is typically less stable.

Light changes the population between these states because each isomer absorbs light differently:
- In the dark, the **trans** isomer is more stable. 
- Irradiation at one wavelength enriches **trans -> cis** (photoisomerization).
- Irradiation at another wavelength can drive the **cis -> trans** (back-isomerization).
- Once in **cis**, and without further irradiating, the system gradually reverts to **trans** (thermal relaxation). 

Mechanistically, thermal and photo-induced isomerization can proceed through rotation or inversion around the azo unit (N=N) (See **Figure 1**). 
- Rotation can be characterized by the CNNC dihedral angle around the azo. 
- Inversion is characterized by the CNN angles

The energy profiles of these relaxation pathways dictate the thermal stability of the **cis** isomer. 
To characterize them using Transition State Theory (TST), one needs to find the minima and the transition states (TS) for each pathway. 

To make matters worse, there is a "triplet" channel, in which the **cis** isomer reverts to **trans** through a rotation-like pathway in which a minimum on the triplet potential energy surface (PES) is explored rather than the rotation TS.
This channel involves two **inter-system crossing (ISC)** steps: S-to-T, T-to-S. It is the energy of those crossing points (and not that of the triplet minimum) what matters in the context of TST.

Also, each ISC step is associated with a probability of hopping, so the actual TST theory needs to be modified, as we will discuss later.

In any case, here we grouped the TS in three types: 
- "**Inversion**": Referring to the inversion pathway in the **singlet** PES
- "**Rotation**":  Referring to the rotation pathway in the **singlet** PES
- "**Triplet**":   Referring to the rotation pathway in the **triplet** PES

<div style="display: flex; justify-content: space-between; align-items: flex-start; gap: 20px;">

  <figure style="width: 48%; margin: 0; text-align: center;">
    <img src="../images/azo_diagram.png" alt="figure1" style="height: 400px; width: auto; max-width: 100%;" />
    <figcaption style="font-size: 0.9em; line-height: 1.4; text-align: justify; margin-top: 6px;">
      <b>Figure 1.</b> Free energy diagram along the reaction coordinate. E and Z correspond to the trans and cis isomers, respectively. The solid red line represents the inversion pathway, while the dark and light green dotted lines represent the rotation pathways on the singlet and triplet potential energy surfaces (PES), respectively.
    </figcaption>
  </figure>

  <figure style="width: 48%; margin: 0; text-align: center;">
    <img src="../images/azo_isc.png" alt="figure2" style="height: 400px; width: auto; max-width: 100%;" />
    <figcaption style="font-size: 0.9em; line-height: 1.4; text-align: justify; margin-top: 6px;">
      <b>Figure 2.</b> Description of the two rotation pathways in the photo-conversion from the Z to the E isomer on the ground singlet state. The black and blue curves describe the singlet (S<sub>0</sub>) and triplet (T<sub>1</sub>) PESs, respectively. The purely singlet pathway (Path<sub>RS</sub>) and the triplet-channel pathway (Path<sub>RT</sub>) are represented by the black and purple curves, respectively. The Path<sub>RT</sub> channel involves intersystem crossing (ISC) cascades (S<sub>0</sub> &rarr; T<sub>1</sub> &rarr; S<sub>0</sub>).
    </figcaption>
  </figure>

</div>

## Part 1. System and State subclasses 

*Recap from Tutorial 1...*

> *A **System** is a collection of chemical entities sharing common properties, typically their chemical composition. 
Its main purpose is to store paths for later use in SCOPE, as well as its **sources***.

Here, we introduce the **System_azo** sub-class, tailored for azo-based molecules. In **part 4** we will show you how to create your azo system from scratch. 

In [ ]:
import os
import numpy as np
import scope
import scope_azo

As we did in Tutorial 1, we start by loading an existing system, called "test":

In [ ]:
## Path of the data folder. It should be "os.path.abspath('../')+'/Data/Azo/1-Tutorial_1/"
data_folder = os.path.abspath('../')+'/Data/9-Tutorial_9/'

## Loads the System object from a binary file, provided in the tutorial folder
sys = scope.load_binary(f"{data_folder}/Systems/test/test.npy")

In [ ]:
## As usual, the Class has a __repr__ method to visualize key information 
print(sys)

In [ ]:
## Notice that it follows the same structure as the core System class.
## Thus, you can use any of the functions defined therein.
sys.set_main_path(data_folder)  ## This is necessary to set the correct path for the data_folder, now that it is in your computer

# Notably, there is one additional attribute that is shown when printing the object: the Dihedral indices
# We will focus on these indices in Part 4 of this tutorial. 
sys.dihedral_indices

### Trans vs. Cis

In [ ]:
## These dihedral indices indicate the atoms involved in the Azo rotation. Lets visualize-them in the "trans" isomer:
found, trans = sys.find_source("trans")
print(found)
trans.view(show_indices=True)

In [ ]:
## This is an azobenzene molecule with two clorine atoms in ortho position, it has two phenyl groups attached to an azo moiety (N=N). 
# Atoms 7 and 8 are the azo itself. 
# Atoms 0 and 9 are its 1st-nearest neighbours. That is, the ring atoms. 
# Atoms 1 and 10 are one of the two possible 2nd-nearest neighbours of the Azo moiety.

## Alltogether, these 6 atoms control the main geometrical features of azobenzene during its photo-isomerization (see introduction, and literature for further details).
## These are important when generating the standard Sources of the System_azo Class, as we will see in Part 3 of this tutorial

In [ ]:
## As usual, we can select its sources for inspection. Here, we select its cis isomer: 
found, cis = sys.find_source("cis")
print(found)

## and visualize it:
cis.view(show_indices=True)

In [ ]:
## Notice that the azo dihedral CNNC has changed.
## To get the actual value, you can use the geometry functions implemented in SCOPE:
P1, P2, P3, P4 = cis.coord[cis.dihedral_indices[1:5]]      ## Store the coordinates of the 4 atoms involved in the CNNC dihedral
radians = scope.geometry.get_dihedral(P1, P2, P3, P4)      ## Computes the dihedral angle in radians
degrees = np.degrees(radians)                              ## Converts to degree
print("CNNC cis:", degrees)

## As we will see later, this is not the optimum structure of the cis isomer, but just its 'initial' structure.

In [ ]:
## If we evaluate the dihedral in the trans isomer...
P1, P2, P3, P4 = trans.coord[trans.dihedral_indices[1:5]]  ## Store the coordinates of the 4 atoms involved in the CNNC dihedral
radians = scope.geometry.get_dihedral(P1, P2, P3, P4)      ## Computes the dihedral angle in radians
degrees = np.degrees(radians)                              ## Converts to degree
print("CNNC trans:", degrees)

## In the trans isomer, the value should approach 180º
## Because of the bulky Cl atoms, the initial structure (generated by open-babel, see azo_functions.get_3D) slightly deviates from planarity

### Transition States

Eight (potential) TS structures can be generated automatically within SCOPE.

In [ ]:
## Here you can find three representative transition states, one for each pathway discussed in the introduction:
found, ts_inv_l_a = sys.find_source("tsinv_l_a")
found, ts_rot_a_s = sys.find_source("tsrot_a_s")
found, ts_rot_a_t = sys.find_source("tsrot_a_t")

## You can uncomment below to visualize them
#ts_inv_l_a.view(show_indices=True)
#ts_rot_a_s.view(show_indices=True)
#ts_rot_a_t.view(show_indices=True)

In [ ]:
## A summary of geometry features is available in the Molecule_azo class: 
ts_inv_l_a.get_geometry_summary()

## Notice the Left Inv. Angle (CNN), corresponding to an Inversion-like structure.

In [ ]:
## This is in contrast with rotation-like structures, which feature a CNNC close to 90º, and CNN and NNC angles close to 113º. 
ts_rot_a_s.get_geometry_summary()


In [ ]:
## These features are also printed in the main __repr__ method of the Molecule_azo class:
ts_rot_a_s

In [ ]:
## Notice that tsrot_a_s is defined as a singlet (spin = 0). The same holds for ts_inv_l_a
print("Spin:", ts_rot_a_s.spin)

In [ ]:
## Instead, the 'triplet' TSs are defined as spin = 2 (as expected)
print(ts_rot_a_t)

## Part 2: Thermal Properties

Here, we discuss the 

```bash
scope run -n scope_env_tutos_azo.npy -s test -i 1-opt_min.scope  2-opt_ts.scope  3-freq_min.scope  4-freq_ts.scope  5-td_min.scope -v -e
```

### Trans vs. Cis stability

In [ ]:
## From the results, one can easily retrieve the Cis - Trans energy difference at a given temperature
sys.get_thermal_stability(target_state='opt', temp=298.15, debug=0) 

### Finding the Minimum Energy Transition State (METS)

In [ ]:
# To find the METS, the function get_mets() goes through all sources with the target state (target_state). 
# - For sources defined in a singlet state, it verifies that it is a TS (one sizeable negative frequency)
# - For sources defined as a triplet, it verifies that it is a minimum (all frequencies are positive, or almost). 
#   Also, it applies an energy penalty to Gtot, to compensate for the ISC processes. 
#   This penalty is applied as dE = - RT ln(p_sh), where p_sh is the probability of surface hopping.
#   Here, p_sh is set to 0.0002, so dE amounts 5 kcal/mol at 298.15 K.
#   This value is a parameter, and can be modified by the user. 
    
## Finally, it selects and returns the source with a lowest Gtot at the requested temperature (temp). That is, the METS
mets = sys.get_mets(temp=298.15, target_state='opt')

In [ ]:
## In this case, the METS is a structure associated with a rotation in the triplet surface
print(mets)

In [ ]:
## The Geometry Features shown above refer to the SOURCE in its INITIAL state. To see those values in the computed target_state (opt), we need to load it:
found, mets_opt_state = mets.find_state("opt")
print(mets_opt_state)

#### Gtot vs. Gtot_eff

In [ ]:
## Beyond geometry, this state is associated with a Gtot value at 298.15K. 
## This value includes vibrational and electronic enthalpy and entropy terms, as discussed in Tutorial 4.
## This are automatically computed during sys.get_mets(), and stored appropriately as State results for each Isomer (cis, trans, ts...)
Gtot_eff = mets_opt_state.results['Gtot_eff']

## Technically, Gtot is an instance of the Collection-class:
print(Gtot_eff)

In [ ]:
## Collections store multiple values associated with the same variable (here, Gtot) but under different circumstances (e.g. temperature). 
## To find the Gtot value at a specific temperature, you can do:
print(Gtot_eff.find_value_with_property('temperature', 298.15))

In [ ]:
# The name 'Gtot_eff' means that any energy penalty has already been considered. 
# In this case, since the METS is a triplet, Gtot_eff differs from the 'original' (i.e. without penalty) Gtot
Gtot = mets_opt_state.results['Gtot'].find_value_with_property('temperature', 298.15)
print(Gtot)
## Notice the slight difference in energy...

In [ ]:
## This difference in energy corresponds to the energy penalty associated with the triplet states
diff = Gtot - Gtot_eff.find_value_with_property('temperature', 298.15)
diff.convert_to_units('kcal')

## In singlet states, Gtot and Gtot_eff should be equal

### Halflife time

In [ ]:
## Overall, with the METS energy, and that of the Trans isomer, one can get obtain the rate constant (k) and the halflife time (t05) of each isomer:
sys.get_cis_halflife_time(temp=298.15, debug=0)

In [ ]:
# The halflife time is stored as a result for this system:
print(sys.results['cis_halflife_time'])

# Together with the thermal relaxation rate
print(sys.results['rate_thermal_cis2trans'])

# and the dG=(G_TS - G_cis)
print(sys.results['dG_from_cis'])

In [ ]:
## This is also possible for the trans isomer: 
sys.get_trans_halflife_time(temp=298.15, debug=0) 

## Part 3. Optical Properties

In [ ]:
# Apart from the cis-trans thermal stability, one of the key properties of Azo complexes is their light-absorption profile.
# For the test compound, the absorption spectrum is simulated using the excited states data obtained from TD-DFT.
# Here, this data is stored in the 'opt' state. 
found, trans_source = sys.find_source('trans')
found, trans_state  = trans_source.find_state('opt')
print(trans_state)

In [ ]:
# The excited states are stored as an attribute:
trans_state.exc_states

### Absorption Spectrum

In [ ]:
# Alltogether, they generate the absorption spectrum
trans_state.plot_abs_spectrum(lmin=200, lmax=400, function='gaussian', sigma=0.2)

# Alternatively, if you prefer the spectrum as x, y arrays, you can run: 
# x, y = trans_state.get_abs_spectrum(lmin=200, lmax=400, function='gaussian', sigma=0.2)

# In both cases, we selected gaussian functions with sigma=0.2 for the convolution. 
# Lorentzian and Laplacian functions can be used instead, as well as other sigma values
trans_state.plot_abs_spectrum(lmin=200, lmax=400, function='lorentzian', sigma='auto')

In [ ]:
# Of course, the same holds for the 'cis' isomer:
found, cis_source = sys.find_source('cis')
found, cis_state  = cis_source.find_state('opt')
trans_state.plot_abs_spectrum(lmin=200, lmax=400, function='gaussian', sigma=0.2)
cis_state.plot_abs_spectrum(lmin=200, lmax=400, function='gaussian', sigma=0.2)

### The Photo-Stationary State (PSS)

Under light irradiation, azo molecules reach a **photostationary state (PSS)** in which both the trans (E) and cis (Z) isomers are populated. The final composition depends on the competition between **thermal relaxation** and **photo-induced isomerization** processes.

Starting from the steady-state condition:

$$
\displaystyle \frac{d[E]}{dt} = - (k^{ph}_{EZ} + k^{th}_{EZ}) [E] + (k^{ph}_{ZE} + k^{th}_{ZE}) [Z] = 0
$$

Then,

$$
\displaystyle k_{EZ}[E] = k_{ZE}[Z]
$$

Within this framework, the fraction -population- of the E isomer under irradiation ($N_E$) can be written as:

$$
\displaystyle N_E = \frac{k_{ZE}}{k_{EZ} + k_{ZE}} =\frac{k^{th}_{ZE} + k^{ph}_{ZE}}
{k^{th}_{EZ} + k^{ph}_{EZ} + k^{th}_{ZE} + k^{ph}_{ZE}}; \quad 
$$

$$
N_Z = 1- N_E 
$$

where $k^{th}_{EZ}$ and $k^{th}_{ZE}$ elements are the **thermal rate constants** for the conversion from trans to cis (EZ) and cis to trans (ZE) while $k_{EZ}^{ph}$ and $k_{ZE}^{ph}$ are the **photoisomerization rate constants**.


The **thermal rates $k^{th}_{EZ}$ and $k^{th}_{ZE}$** are normally evaluated using Eyring equation, or obtained from experimental half-life times, $t_{1/2}$ assuming a first-order reaction: 

$$k^{th} = \frac{ln(2)}{t_{1/2}} \quad [s^{-1}]$$


The **photochemical rates $k^{ph}_{EZ}$ and $k^{ph}_{ZE}$** can be modeled as

$$
\displaystyle k^{ph}_{EZ} =
\phi_{EZ} \int \sigma_E(\lambda)\,\Phi(\lambda)\, d\lambda \quad [s^{-1}]

$$

by combining three physical quantities:

- the **absorption probability** of the molecule (or absorption cross section): $\sigma_E(\lambda)$ &emsp;&emsp;[$m^2$]
- the **number of incoming photons** (or photon flux): $\Phi(\lambda)$ &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;[$photons·m^{-2} s^{-1} nm^{-1}$]
- the **probability that absorption leads to reaction** (or quantum yield): $\phi_{EZ}$ &emsp;&emsp;&emsp;&emsp;&nbsp; [$a.u$]

More specifically:

- $\Phi(\lambda)$ represents the number of photons reaching the sample per unit area, per unit time, and per wavelength interval. It is obtained from the irradiance divided by the energy of a single photon:

$$
\Phi(\lambda) = \frac{I(\lambda)}{E_{photon}};
\quad \frac{\left[\frac{J}{s·m^2 nm^{-1}}\right]}{\left[ J \right]}
$$

- $\sigma_E(\lambda)$ quantifies the probability that a molecule absorbs a photon of wavelength $\lambda$. **Note:** *The spectra obtained by using get_abs_spectrum() returns the oscillator strengths vs. wavelength, $f(\lambda)$. To provide physical meaning (by converting it to an absorption cross section), the y-axis is converted to* $m^2$ *by applying a factor K (note that a $s^{-1}$ should appear, and it is cancelled by frequency units implicit in the wavelength space):*

$$
\sigma_E(\lambda) = \frac{\pi · h · e}{ \varepsilon _0 · m_e · c } * f(\lambda) \quad \left[ m^2 \right]
$$

- $\phi_{ZE}$ measures the efficiency of the photochemical reaction. It corresponds to the fraction of absorbed photons that result in isomerization (for example, $\phi_{ZE}=0.5$ means that half of the absorbed photons lead to reaction).

In practice, light sources are not perfectly monochromatic (even lasers emit photons across a finite wavelength range). For this reason, contributions from all wavelengths must be considered, which is achieved through the integral above. The resulting equation corresponds to the **total absorption rate per molecule** and can be interpreted as a spectral overlap between the lamp emission spectrum and the molecular absorption spectrum.

Once the populations in the steady state, and $N_E$,$N_Z$ at an given irradiation wavelength are known, it is possible to obtain PSS absorption cross section spectrum 
$$
\displaystyle \sigma_{PSS}(\lambda) = N_E  \sigma_{E}(\lambda)+ N_Z \sigma_{Z}(\lambda)
$$

along with the molar absorptivity, $\varepsilon$ in $L · mol^{-1}·cm^{-1}$, applying the Lambert-Beer concentration factors

$$
\displaystyle \varepsilon_{PSS}(\lambda) = \frac{N_A}{1000 · ln(10)} \sigma_{PSS}(\lambda)
$$

Then, the molar absorptivity at the steady state can be represented graphically for each irradiation wavelength 
$\displaystyle \varepsilon_{PSS}(\lambda)$  vs. $\lambda$.  

#### The 'Lamp' Class

In [ ]:
from scope_azo.azo_classes import Lamp

In [ ]:
# The properties of the light source are provided to the PSS as a Lamp-Class object.
# Lamps store information about the available wavelengths, and associated FWHM and Power values.
# If you don't have such information, you can use a "default" Lamp that we provide within SCOPE.
# Any Lamp can be tuned at will by the user, and new Lamps can also be added as simple JSON files. 
my_lamp = Lamp("default")
print(my_lamp)

In [ ]:
# Basically, this Lamp provides 12 excitation wavelentghs defined as:
# - gaussians of FWHM=20 nm
# - with 50 mW of power when used at 100% (Power Intensity=1.0). Can be changed with my_lamp.set_power_intensity()

In [ ]:
# For the moment, only one real-world device has been included, the CoolLED pE-4000:
my_lamp = scope_azo.azo_classes.Lamp("coolled")
print(my_lamp) 

In [ ]:
# Finally, absence of irradiation can be simulated with the "dark" Lamp:
my_lamp = scope_azo.azo_classes.Lamp("dark")
print(my_lamp) 

In [ ]:
# To compute the PSS, one just needs to use the get_PSS() function as:
my_pss = sys.get_PSS(lamp_name='default', target_state='opt', temp=298.15, lmin=200, lmax=600)

# This function has several options and defaults, do not forget to read the docs if necessary:
#help(sys.get_PSS)

# In summary, the function takes/computes all relevant data from the stored sources (trans, cis), and creates a PSS-class instance:
print(my_pss)

# This class stores relevant data to compute the PSS at any given wavelength, such as the thermal rates.

In [ ]:
# Wavelength-dependent results are stored as a dictionary, with wavelengths as keys:
res = my_pss.pss_results[250]

print(res['wavelength'])                ## Wavelength of irradiation
print(res['pss_trans_ratio'])           ## Ratio of Trans Isomer in the PSS
print(res['rate_photo_trans2cis'])      ## Photoisomerization Rate for E-->Z at this wavelength 
print(res['rate_photo_cis2trans'])      ## Photoisomerization Rate for Z-->E at this wavelength   
print(res['rate_thermal_trans2cis'])    ## Thermal Reaction Rate for E-->Z (independent of wavelength) 
print(res['rate_thermal_cis2trans'])    ## Thermal Reaction Rate for Z-->E (independent of wavelength) 
print(res['QY_trans2cis'])              ## Quantum Yield for E-->Z photoisomerization at this wavelength  
print(res['QY_cis2trans'])              ## Quantum Yield for Z-->E photoisomerization at this wavelength


In [ ]:
# Results can be plotted with:
my_pss.plot_spectra(lmax=500)

In [ ]:
# Above, we can see that we only obtain a mixture of E and Z isomers under light between 250-340 nm. 
# For all other cases, only the trans state is present

# These are, unfortunately, boring results from the chemistry point of view. 

## Part 4. Creating new Azo-based Systems

Now that you have explored some of the useful features to carry out your computational experiments, let's see how can you create your own dataset of azoheteroarene species.

This **Part 4** shows how to create a new system from scratch, and feed it with **sources**. Every step is quite automatic, and nothing should go wrong if you follow the instructions shown below. 

In [ ]:
# First, let's define a SMILES string. It is important to be careful when 
#   constructing your SMILES: it MUST follow the pattern below to correctly detect
#   atom indices near the Azo group, enabling rotation and inversion of both rings.

#   IMPORTANT: SMILES strings MUST adhere to the following structure:
#                    'C1(...C1)/N=N/ring2' 
#                              or
#                    'C1(...C1)/N=N\\ring2' 


# We will select azobenzene for its simplicity. Let's store the SMILES string and its name.
 
smiles = 'c1(ccccc1)/N=N\\c2ccccc2'      # It doesn't matter if the SMILES string is in the cis or trans way. 
name = 'Azobenzene' 


In [ ]:
# We will create our system as a System_azo object providing its name and the smiles. 
azo_sys = scope_azo.System_azo(name, smiles)
print(azo_sys)

### Working with System_azo Objects

In [ ]:
# System_azo.create_trans()
    # Creates trans structures.
    # It uses the SMILES string stored in the System to generate 3D coordinates.
    # If everything works, it adds the trans isomer as a `Molecule_azo` object as source of the system 
    # Returns the `Molecule_azo` object.

In [ ]:
# Let's create the trans isomer. It is automatically registered as a source in our azobenzene system.
# Note that the 'overwrite' parameter defaults to False. 
# If Openbabel is not installed, it will raise and error. Install it with 'conda install openbabel -c conda-forge'
trans = azo_sys.create_trans(overwrite=True)
print(trans)

In [ ]:
# The 3D structure of the models is created with Openbabel, and is accessible through:

print(trans.labels)
print(trans.coord)

In [ ]:
# System_azo.create_cis(target_deg=40.)
    # Creates cis structures from the trans isomer. if the trans isomer is not created, it is, too.
    # If everything works, it adds the cis isomer as a `Molecule_azo` object as source of the system.
    # The `target_deg` parameter allows you to choose the torsion of the dihedral angle of the Azo group (CN=NC). 
    # Please, insert your target angle in degrees.

In [ ]:
# The same with the cis isomer. It is created with create_cis() method. 
# By default, the azo group dihedral angle is set to 40.0 degrees. 
# It can be changed by modifying "target_deg"
cis_iso = azo_sys.create_cis(target_deg=30.)
print(cis_iso)

### Creation of transition states

In [ ]:
# System_azo.create_ts(ts_list=['TSrot', 'TSinv', 'triplet'])
#
# The `ts_list` parameter allows you to specify which transition state 
# (TS) structures to generate:
#
# - TSrot: Generates the rotation of the azo group. It creates two 
#   structures in the singlet state with the central C-N=N-C dihedral 
#   angle set to +90 deg (TSrot_a_s) and -90 deg (TSrot_b_s).
#
# - TSinv_l & TSinv_r: Generates the inversion of the left (TSinv_l) 
#   or right (TSinv_r) ring. 
#     * The angle of the inverting nitrogen (C-N=N for the left 
#       ring, or N=N-C for the right ring) is linearized to 180 deg.
#     * Each inversion pathway creates two variants (_a and _b) 
#       corresponding to the dihedral angle between the inverting ring 
#       and the azo group, set to 0 deg and 180 deg respectively 
#       (e.g., TSinv_r_a and TSinv_r_b).
#
# - triplet: If included, this flag also generates the rotational 
#   transition states (TSrot) in the triplet state by setting the 
#   spin to 2.

In [ ]:
# You can read the docstring of each function for more information.

In [ ]:
# Finally, we can generate the Transition State (TS) geometries. 
# These are returned as a list and automatically registered as sources in the System_azo.

ts_list = azo_sys.create_ts()
print([ts.name for ts in ts_list])

# Pay attention to the naming convention: 
# Since all TSs are generated, the singlet and triplet states are labeled 
# 'TSrot_A_S' and 'TSrot_A_T', respectively.

In [ ]:
# And we can check if geometries are correct using the _repr_ method included for Molecule_azo species...
ts = azo_sys.find_source('tsrot_a_s')[1]
print(ts)

In [ ]:
# ... or using the method Specie.view()
ts.view(show_indices=True)

# Check the Azo group dihedral angle is set up at 90º

In [ ]:
# These Molecules_azo store labels and coordinates of the geometry stored in the 'initial' state.
import numpy as np
trans = azo_sys.find_source('trans')[1]
initial_state = trans.find_state('initial')[1]
print('Coordinates are equal: ', np.array_equal(initial_state.coord,trans.coord))

In [ ]:
# Let's inspect the relevant atom indices for the trans isomer by accessing the 
# 'dihedral_indices' attribute:

print('Atom Indices for Dihedral: ', azo_sys.dihedral_indices)

# Recall that the atoms involved in the C-N=N-C rotation correspond to indices 0, 6, 7, and 8.
# These represent the four atoms extracted from the index list defining the torsion.

# The first and last elements of this list correspond to the anchor atoms of the 
# left and right rings, respectively.

# The variables 'atom0' and 'atom1' will be used to facilitate independent ring rotation.

atom0, atom1, atom2, atom3, atom4, atom5 = azo_sys.dihedral_indices

In [ ]:
# These indices allow us to analyze geometric parameters, such as the dihedral angle defined by the C-N=N-C ...
coord = trans.coord
dih_angle = np.degrees(scope.geometry.get_dihedral(coord[atom1], coord[atom2], coord[atom3], coord[atom4]))
print(f'{dih_angle:.2f} degrees')

In [ ]:
# ... or the angle between CNN atoms. Here is the angle defined by the CN=N atoms. 
angle = np.degrees(scope.geometry.get_angle(coord[atom1] - coord[atom2], coord[atom2] - coord[atom3]))
print(f'{angle:.2f} degrees')

In [ ]:
# Using the core SCOPE module, modifying dihedral angles is straightforward:

rotated_coord = scope.geometry.set_dihedral(trans.labels, coord, 30, atom1, atom2, atom3, atom4)
new_molec     = scope_azo.Molecule_azo(trans.labels, rotated_coord)
new_molec.view(show_indices=True)

# Notice the steric clashes involving atoms 5, 13, 18, and 23. 
# Don't worry—we have a specific solution to resolve this.

In [ ]:
# The create_cis() method automatically resolves steric clashes when rotating 
# to a specific angle, defined by the 'target_deg' parameter.

# We recommend using this function for geometries prone to collisions (steric hindrance),
# as it automatically adjusts adjacent rings to achieve a valid, clash-free conformation.
azo_sys.create_cis(target_deg=30, overwrite=True).view()

> **Pro Tip:** We recommend using `create_cis()` for geometries prone to steric hindrance. It automatically adjusts adjacent rings to ensure a valid, clash-free conformation.

### Set Paths and calculations 

In [ ]:
# Now, let's configure the directory paths. 
# You can use the interactive azo_sys.set_paths() method (ideal for terminal sessions) 
# or define them manually. Below, we demonstrate the explicit approach.
 
tutorial_folder = os.path.abspath('../')+'/Data/Azo/'
print(tutorial_folder)

# Defining paths explicitly:
azo_sys.set_main_path(f"{tutorial_folder}")
azo_sys

In [ ]:
# The system can be saved to a .npy binary using 'load_binary()' function.
# As we have just assigned the system filepath, saving the binary becomes easy.

scope.save_binary(azo_sys, azo_sys.system_file)

## Part 2. Going further: Combining smiles. (WIP)

In [ ]:
# In computational studies, it is common to study multiple systems simultaneously.

# To speed up this, we allow you to construct systems by combining specific ring structures.
# You simply need to provide the SMILES for each ring (including substituents), adhering to 
# the pattern shown above:
#                           C1(...C1)/N=N/ring2 

# From now on, we will refer to these rings as 'fragments'.

# As an example, let's define two fragments for the left ring and one for the right.
# IMPORTANT: Each fragment MUST be a string stored within a list.

lefts   = ['C1=(C-C=C-C=C1)','C1=(CC=NC=C1)']
rights  = ['c2ccccc2']



In [ ]:
# The combine_smiles() function returns a list containing every combination of the fragments. 
# You can optionally provide a filename to save the output to a text file.

combine_smiles(lefts,rights)

# NO FUNCIONA ENCARA 

In [ ]:
azoben_sys